# AntiDeepFake-Face (M5) — Train & Evaluate Deepfake Detector on Google Colab (GPU L4)

**Phụ trách:** M5 — Lê Minh Thành, Nguyễn Phông  
**Bộ dữ liệu:** `dataset_RTFS_800_Subset` (800 video, 4.800 crop khuôn mặt `224×224` MTCNN, chia **0 rò rỉ danh tính** `train=480 / val=120 / test=200`).  
**Mục tiêu:**
1. Đánh giá Baseline **Zero-shot Pretrained** (`king1oo1/deepfake-model` và `prithivMLmods/Deep-Fake-Detector-v2-Model`) trên tập `test` giữ kín (E5).
2. Fine-tune bằng **GPU NVIDIA L4 (BF16)** theo kỹ thuật **GenD-style LayerNorm + Top-4 Transformer Blocks** (và so sánh thêm CNN nhẹ `efficientnet_b0`) để kéo **ROC-AUC** từ `~0.62–0.66` lên `>0.85–0.92+` trong khi giữ nguyên tốc độ suy luận CPU laptop (`≥ 11 FPS`).
3. Xuất file adapter `m5_rtfs_siglip2_adapter.pt` + kết quả JSON/CSV/biểu đồ về Google Drive để thả trực tiếp vào `models/` và `results/` trên máy local.

### Cách chạy nhanh trên Google Colab:
1. Vào **Runtime → Change runtime type → L4 GPU** (Python 3).
2. Upload file `data/cache/rtfs_m5_colab_dataset.zip` từ máy local lên Google Drive (mặc định đặt tại `MyDrive/AntiDeepFake-Face/rtfs_m5_colab_dataset.zip`).
3. Chỉnh đường dẫn `DRIVE_ZIP_PATH` ở **Cell 2** nếu bạn lưu chỗ khác, rồi bấm **Runtime → Run all**.

In [ ]:
# 1. Kiểm tra GPU NVIDIA L4 và cài thư viện cần thiết
import torch, sys
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Chưa bật GPU! Vào Runtime -> Change runtime type -> L4 GPU."
gpu_name = torch.cuda.get_device_name(0)
bf16_ok = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu_name} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB | BF16: {bf16_ok}")

%pip install -q "transformers>=4.45.0" timm scikit-learn safetensors pandas matplotlib

In [ ]:
# 2. Kết nối Google Drive & giải nén dataset ra SSD cục bộ /content/ (đọc nhanh gấp 20x)
import os, zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# === CHỈNH ĐƯỜNG DẪN NẾU BẠN LƯU Ở THƯ MỤC KHÁC TRÊN DRIVE ===
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/AntiDeepFake-Face/rtfs_m5_colab_dataset.zip")
DRIVE_OUT_DIR  = Path("/content/drive/MyDrive/AntiDeepFake-Face/m5_colab_outputs")
LOCAL_DATA_DIR = Path("/content/rtfs_m5_colab_dataset")

assert DRIVE_ZIP_PATH.is_file(), (
    f"Không tìm thấy {DRIVE_ZIP_PATH}. Hãy upload file data/cache/rtfs_m5_colab_dataset.zip lên Drive trước!"
)
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not (LOCAL_DATA_DIR / "rtfs_m5_rich_metadata.csv").is_file():
    print(f"Đang giải nén {DRIVE_ZIP_PATH} -> /content/ ...")
    with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as zf:
        zf.extractall("/content")
print("Dataset sẵn sàng tại:", LOCAL_DATA_DIR)

In [ ]:
# 3. Kiểm tra toàn vẹn dữ liệu & xác minh 0 rò rỉ subject_id giữa train / val / test
import pandas as pd

meta_df = pd.read_csv(LOCAL_DATA_DIR / "rtfs_m5_rich_metadata.csv")
print("Tổng số video:", len(meta_df))
display(meta_df.groupby(["split", "label", "generator", "has_faceenhancer"]).size().unstack(fill_value=0))

# Kiểm tra tuyệt đối không trùng subject_id giữa bất kỳ cặp tập nào
splits = ["train", "val", "test"]
for i in range(len(splits)):
    for j in range(i + 1, len(splits)):
        a, b = splits[i], splits[j]
        sa = set(meta_df.loc[meta_df["split"] == a, "subject_id"])
        sb = set(meta_df.loc[meta_df["split"] == b, "subject_id"])
        assert len(sa & sb) == 0, f"Phát hiện rò rỉ {a} vs {b}: {len(sa & sb)} subject_id!"
        print(f"Leakage check ({a} vs {b}): 0 trùng subject_id | {a}={len(sa)} subjects, {b}={len(sb)} subjects")

# Đếm ảnh PNG theo từng thư mục
for sp in splits:
    n_real = len(list((LOCAL_DATA_DIR / sp / "Real").glob("*.png")))
    n_fake = len(list((LOCAL_DATA_DIR / sp / "Deepfake").glob("*.png")))
    print(f"  [{sp:5s}] PNG crops -> Real: {n_real:4d}, Deepfake: {n_fake:4d}, Total: {n_real + n_fake:4d}")

### Phân tích dữ liệu đầu vào
- Bộ dữ liệu có tỷ lệ lớp **1:3** (`25% Real` / `75% Deepfake`) đồng nhất trên cả 3 tập `train`, `val`, `test`, và đã được xác minh **0 trùng `subject_id`** giữa mọi cặp tập.
- Để xử lý mất cân bằng lớp và ngăn mô hình học vẹt nén video/ánh sáng, bước huấn luyện bên dưới áp dụng **trọng số lớp cân bằng (`3:1`) + Label Smoothing (`0.05`)** và **Forensic Augmentation** (lật ngang, random crop nhẹ, đổi sáng/tương phản, mô phỏng nén JPEG `Q=55..95`, Gaussian blur).

In [ ]:
# 4. Định nghĩa Dataset, Forensic Augmentation & bộ chỉ số đánh giá chuẩn E5 (ISO/IEC 30107-3)
import io, json, random, time, copy
from typing import Any, Dict, List, Sequence, Tuple
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, roc_auc_score, roc_curve
)

SEED = 42
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda")
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

_FAKE_TOKENS = {"fake", "deepfake", "deepfakes", "ai", "generated", "synthetic", "manipulated", "forged", "spoof"}
_REAL_TOKENS = {"real", "realism", "authentic", "genuine", "human", "original", "bonafide"}

def resolve_fake_label_index(id2label: Dict[Any, str]) -> int:
    norm = {int(k): str(v).strip().lower() for k, v in id2label.items()}
    fake_ids = [idx for idx, name in norm.items() if any(t in name for t in _FAKE_TOKENS)]
    if len(fake_ids) == 1:
        return fake_ids[0]
    real_ids = [idx for idx, name in norm.items() if any(t in name for t in _REAL_TOKENS)]
    if len(norm) == 2 and len(real_ids) == 1:
        return [i for i in norm if i != real_ids[0]][0]
    raise ValueError(f"Không xác định được lớp Fake từ id2label={id2label}")

def apply_forensic_aug(img: Image.Image) -> Image.Image:
    if random.random() < 0.5:
        img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    if random.random() < 0.65:
        w, h = img.size
        scale = random.uniform(0.82, 1.0)
        nw, nh = int(w * scale), int(h * scale)
        x0 = random.randint(0, max(0, w - nw))
        y0 = random.randint(0, max(0, h - nh))
        img = img.crop((x0, y0, x0 + nw, y0 + nh)).resize((w, h), Image.Resampling.BILINEAR)
    if random.random() < 0.6:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.80, 1.20))
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.80, 1.20))
        img = ImageEnhance.Color(img).enhance(random.uniform(0.85, 1.15))
    if random.random() < 0.25:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.2)))
    if random.random() < 0.65:
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(55, 95))
        buf.seek(0)
        with Image.open(buf) as dec:
            img = dec.convert("RGB")
    return img

class RTFSCropDataset(Dataset):
    def __init__(self, split_dir: Path, mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), image_size=224, augment=False):
        self.samples: List[Tuple[Path, int, str]] = []
        for label_name, y in [("Real", 0), ("Deepfake", 1)]:
            for p in sorted((split_dir / label_name).glob("*.png")):
                sample_id = p.name.split("__")[0]
                self.samples.append((p, y, sample_id))
        self.mean = np.asarray(mean, dtype=np.float32).reshape(3, 1, 1)
        self.std = np.asarray(std, dtype=np.float32).reshape(3, 1, 1)
        self.image_size = image_size
        self.augment = augment

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, y, sid = self.samples[idx]
        with Image.open(path) as im:
            img = im.convert("RGB")
        if self.augment:
            img = apply_forensic_aug(img)
        if img.size != (self.image_size, self.image_size):
            img = img.resize((self.image_size, self.image_size), Image.Resampling.BILINEAR)
        arr = np.asarray(img, dtype=np.float32) / 255.0
        chw = (np.transpose(arr, (2, 0, 1)) - self.mean) / self.std
        return torch.from_numpy(chw), y, sid

def bootstrap_auc_ci(y_true, y_score, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(float(roc_auc_score(y_true[idx], y_score[idx])))
    return [round(float(np.percentile(aucs, 2.5)), 4), round(float(np.percentile(aucs, 97.5)), 4)]

def find_optimal_threshold(y_true, y_score):
    fpr, tpr, thrs = roc_curve(y_true, y_score)
    acers = 0.5 * (fpr + (1.0 - tpr))
    best_i = int(np.argmin(acers))
    eer_i = int(np.argmin(np.abs(fpr - (1.0 - tpr))))
    thr = float(thrs[best_i])
    return (thr if np.isfinite(thr) else 0.5), float(0.5 * (fpr[eer_i] + (1.0 - tpr[eer_i])))

def metrics_at_thr(y_true, y_score, thr):
    y_pred = (y_score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    apcer = float(fn / (tp + fn)) if (tp + fn) > 0 else 0.0
    bpcer = float(fp / (tn + fp)) if (tn + fp) > 0 else 0.0
    return {
        "threshold": round(float(thr), 4),
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "balanced_accuracy": round(float(balanced_accuracy_score(y_true, y_pred)), 4),
        "f1_macro": round(float(f1_score(y_true, y_pred, average="macro", zero_division=0)), 4),
        "apcer": round(apcer, 4),
        "bpcer": round(bpcer, 4),
        "acer": round(0.5 * (apcer + bpcer), 4),
        "cm": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }

def evaluate_split(df_split: pd.DataFrame, vid_scores: Dict[str, float], threshold: float, with_ci=True):
    yt = (df_split["label"] == "Deepfake").astype(int).to_numpy()
    ys = np.asarray([vid_scores[sid] for sid in df_split["sample_id"]], dtype=float)
    auc = float(roc_auc_score(yt, ys))
    _, eer = find_optimal_threshold(yt, ys)

    def _slice(mask):
        sub = df_split[mask]
        s_yt = (sub["label"] == "Deepfake").astype(int).to_numpy()
        s_ys = np.asarray([vid_scores[sid] for sid in sub["sample_id"]], dtype=float)
        if len(np.unique(s_yt)) < 2:
            return {"n": len(sub)}
        m = metrics_at_thr(s_yt, s_ys, threshold)
        return {"n": len(sub), "roc_auc": round(float(roc_auc_score(s_yt, s_ys)), 4), "acer": m["acer"], "recall_fake": round(1.0 - m["apcer"], 4)}

    enh = df_split["has_faceenhancer"].astype(str) == "True"
    slices = {
        "inswapper_all": _slice(df_split["generator"].isin(["original", "inswapper"])),
        "inswapper_no_enh": _slice((df_split["generator"] == "original") | ((df_split["generator"] == "inswapper") & (~enh))),
        "inswapper_with_enh": _slice((df_split["generator"] == "original") | ((df_split["generator"] == "inswapper") & enh)),
        "uniface_all": _slice(df_split["generator"].isin(["original", "uniface"])),
        "uniface_no_enh": _slice((df_split["generator"] == "original") | ((df_split["generator"] == "uniface") & (~enh))),
        "uniface_with_enh": _slice((df_split["generator"] == "original") | ((df_split["generator"] == "uniface") & enh)),
        "lighting_bright": _slice(df_split["lighting"] == "bright"),
        "lighting_average": _slice(df_split["lighting"] == "average"),
        "lighting_dark": _slice(df_split["lighting"] == "dark"),
    }
    return {
        "roc_auc": round(auc, 4),
        "ci95": bootstrap_auc_ci(yt, ys) if with_ci else [round(auc, 4), round(auc, 4)],
        "pr_auc": round(float(average_precision_score(yt, ys)), 4),
        "eer": round(eer, 4),
        "at_val_tau": metrics_at_thr(yt, ys, threshold),
        "at_0_50": metrics_at_thr(yt, ys, 0.50),
        "slices": slices,
    }

@torch.inference_mode()
def predict_video_scores(model: nn.Module, loader: DataLoader, fake_idx: int, is_hf: bool = True) -> Dict[str, float]:
    model.eval()
    buckets: Dict[str, List[float]] = {}
    for x, _y, sids in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE):
            out = model(pixel_values=x).logits if is_hf else model(x)
            probs = torch.softmax(out.float(), dim=-1)[:, fake_idx]
        for sid, p in zip(sids, probs.cpu().tolist()):
            buckets.setdefault(sid, []).append(float(p))
    return {sid: float(np.mean(v)) for sid, v in buckets.items()}

df_val = meta_df[meta_df["split"] == "val"].reset_index(drop=True)
df_test = meta_df[meta_df["split"] == "test"].reset_index(drop=True)
val_y_vid = (df_val["label"] == "Deepfake").astype(int).to_numpy()
print("Utilities ready.")

In [ ]:
# 5. Hàm huấn luyện GenD-style (All LayerNorms + Top-K Transformer Blocks + Classifier) trên GPU L4
from transformers import AutoImageProcessor, AutoModelForImageClassification

def get_norm_stats(processor):
    mean = tuple(getattr(processor, "image_mean", (0.5, 0.5, 0.5)) or (0.5, 0.5, 0.5))
    std = tuple(getattr(processor, "image_std", (0.5, 0.5, 0.5)) or (0.5, 0.5, 0.5))
    size_cfg = getattr(processor, "size", 224)
    if isinstance(size_cfg, dict):
        img_sz = size_cfg.get("height") or size_cfg.get("shortest_edge") or 224
    elif hasattr(size_cfg, "height") or hasattr(size_cfg, "shortest_edge"):
        img_sz = getattr(size_cfg, "height", None) or getattr(size_cfg, "shortest_edge", None) or 224
    elif isinstance(size_cfg, (int, float)):
        img_sz = int(size_cfg)
    else:
        img_sz = 224
    return mean, std, int(img_sz)

def configure_trainable_params(model: nn.Module, unfreeze_top_k: int = 4):
    """Mở khóa toàn bộ LayerNorm (GenD) + Top-K block cuối + Classifier head."""
    for p in model.parameters():
        p.requires_grad = False

    trainable_names = set()
    # 1. Mở khóa toàn bộ LayerNorm trong toàn mạng (GenD PEFT)
    for name, mod in model.named_modules():
        if isinstance(mod, nn.LayerNorm):
            for pn, p in mod.named_parameters(recurse=False):
                p.requires_grad = True
                trainable_names.add(f"{name}.{pn}")

    # 2. Mở khóa Top-K Transformer blocks cuối cùng
    layers = None
    layer_prefix = ""
    if hasattr(model, "vision_model") and hasattr(model.vision_model, "encoder"):
        layers = model.vision_model.encoder.layers
        layer_prefix = "vision_model.encoder.layers"
    elif hasattr(model, "vit") and hasattr(model.vit, "layers"):
        layers = model.vit.layers
        layer_prefix = "vit.layers"

    if layers is not None and unfreeze_top_k > 0:
        start_idx = max(0, len(layers) - unfreeze_top_k)
        for idx in range(start_idx, len(layers)):
            for pn, p in layers[idx].named_parameters():
                p.requires_grad = True
                trainable_names.add(f"{layer_prefix}.{idx}.{pn}")

    # 3. Mở khóa Classification Head
    if hasattr(model, "classifier"):
        for pn, p in model.classifier.named_parameters():
            p.requires_grad = True
            trainable_names.add(f"classifier.{pn}")

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {n_train/1e6:.2f}M / {n_total/1e6:.2f}M ({100*n_train/n_total:.1f}%)")
    return trainable_names

def run_hf_experiment(
    tag: str,
    model_id: str,
    unfreeze_top_k: int = 4,
    epochs: int = 10,
    batch_size: int = 64,
    lr_blocks: float = 2e-5,
    lr_ln: float = 1e-4,
    lr_head: float = 3e-4,
):
    print(f"\n==================== {tag.upper()} ({model_id}) ====================")
    set_seed(SEED)
    processor = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModelForImageClassification.from_pretrained(model_id).to(DEVICE)
    id2label = {int(k): str(v) for k, v in model.config.id2label.items()}
    fake_idx = resolve_fake_label_index(id2label)
    real_idx = 1 - fake_idx
    mean, std, img_sz = get_norm_stats(processor)
    print(f"  id2label={id2label} | fake_idx={fake_idx} | size={img_sz} | mean={mean}")

    ds_train = RTFSCropDataset(LOCAL_DATA_DIR / "train", mean=mean, std=std, image_size=img_sz, augment=True)
    ds_val   = RTFSCropDataset(LOCAL_DATA_DIR / "val",   mean=mean, std=std, image_size=img_sz, augment=False)
    ds_test  = RTFSCropDataset(LOCAL_DATA_DIR / "test",  mean=mean, std=std, image_size=img_sz, augment=False)

    dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True, drop_last=False)
    dl_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    dl_test  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # A. Zero-shot baseline (chưa train)
    va_zs = predict_video_scores(model, dl_val, fake_idx, is_hf=True)
    te_zs = predict_video_scores(model, dl_test, fake_idx, is_hf=True)
    zs_tau, _ = find_optimal_threshold(val_y_vid, np.asarray([va_zs[sid] for sid in df_val["sample_id"]]))
    zs_res = {
        "model_id": model_id,
        "method": "Zero-shot Pretrained",
        "val_tau": round(zs_tau, 4),
        "val": evaluate_split(df_val, va_zs, zs_tau, with_ci=False),
        "test": evaluate_split(df_test, te_zs, zs_tau, with_ci=True),
    }
    print(f"  [Zero-shot] val_AUC={zs_res['val']['roc_auc']} | TEST_AUC={zs_res['test']['roc_auc']} {zs_res['test']['ci95']} | TEST_ACER={zs_res['test']['at_val_tau']['acer']}")

    # B. Fine-tune trên GPU L4 (All LayerNorms + Top-K Blocks + Classifier)
    trainable_names = configure_trainable_params(model, unfreeze_top_k=unfreeze_top_k)

    head_params, ln_params, block_params = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("classifier."):
            head_params.append(p)
        elif "layernorm" in name.lower() or "layer_norm" in name.lower() or "norm" in name.lower():
            ln_params.append(p)
        else:
            block_params.append(p)

    optimizer = torch.optim.AdamW(
        [
            {"params": block_params, "lr": lr_blocks},
            {"params": ln_params, "lr": lr_ln},
            {"params": head_params, "lr": lr_head},
        ],
        weight_decay=0.05,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    # Class weight cân bằng 3:1 (Real:Fake) + label smoothing
    weights = torch.zeros(2, device=DEVICE, dtype=torch.float32)
    weights[real_idx] = 2.0
    weights[fake_idx] = 2.0 / 3.0
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

    best_val_auc = -1.0
    best_adapter_sd = None
    best_va_ft = None
    history = []

    for ep in range(1, epochs + 1):
        t0 = time.perf_counter()
        model.train()
        total_loss, n_steps = 0.0, 0
        for x, y_bin, _ in dl_train:
            x = x.to(DEVICE, non_blocking=True)
            y_native = torch.where(y_bin == 1, fake_idx, real_idx).to(DEVICE, dtype=torch.long, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(pixel_values=x).logits
                loss = criterion(logits, y_native)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += float(loss.item())
            n_steps += 1
        scheduler.step()

        va_scores = predict_video_scores(model, dl_val, fake_idx, is_hf=True)
        va_auc = float(roc_auc_score(val_y_vid, [va_scores[sid] for sid in df_val["sample_id"]]))
        va_thr, va_eer = find_optimal_threshold(val_y_vid, np.asarray([va_scores[sid] for sid in df_val["sample_id"]]))
        is_best = va_auc > best_val_auc
        if is_best:
            best_val_auc = va_auc
            best_va_ft = va_scores
            best_adapter_sd = {
                k: v.detach().cpu().to(torch.float16).clone()
                for k, v in model.state_dict().items()
                if k in trainable_names
            }
        history.append({"epoch": ep, "loss": round(total_loss / max(1, n_steps), 4), "val_auc": round(va_auc, 4), "val_eer": round(va_eer, 4)})
        print(f"    ep {ep:02d}/{epochs} | loss={total_loss/n_steps:.4f} | val_AUC={va_auc:.4f} | val_EER={va_eer:.4f} | {time.perf_counter()-t0:.1f}s{' *' if is_best else ''}")

    # Nạp lại best checkpoint theo val AUC để chấm 1 lần duy nhất trên test
    cast_best = {k: v.to(device=DEVICE, dtype=next(model.parameters()).dtype) for k, v in best_adapter_sd.items()}
    model.load_state_dict(cast_best, strict=False)
    te_ft = predict_video_scores(model, dl_test, fake_idx, is_hf=True)
    ft_tau, _ = find_optimal_threshold(val_y_vid, np.asarray([best_va_ft[sid] for sid in df_val["sample_id"]]))

    # Lưu adapter tương thích 100% với src/deepfake/predictor.py
    adapter_file = DRIVE_OUT_DIR / f"m5_rtfs_{tag}_adapter.pt"
    torch.save(
        {
            "format": "adf_m5_partial_adapter_v1",
            "model_id": model_id,
            "arch": tag,
            "unfreeze_top_k": unfreeze_top_k,
            "id2label": id2label,
            "fake_label_index": fake_idx,
            "recommended_threshold": round(float(ft_tau), 4),
            "state_dict": best_adapter_sd,
        },
        adapter_file,
    )
    adapter_mb = round(adapter_file.stat().st_size / (1024 * 1024), 2)

    ft_res = {
        "model_id": model_id,
        "method": f"GenD-LN + Top-{unfreeze_top_k} Blocks Fine-tune (L4 BF16)",
        "adapter_file": adapter_file.name,
        "adapter_size_mb": adapter_mb,
        "val_tau": round(ft_tau, 4),
        "history": history,
        "val": evaluate_split(df_val, best_va_ft, ft_tau, with_ci=False),
        "test": evaluate_split(df_test, te_ft, ft_tau, with_ci=True),
    }
    print(f"  [Fine-tuned] val_AUC={ft_res['val']['roc_auc']} | TEST_AUC={ft_res['test']['roc_auc']} {ft_res['test']['ci95']} | TEST_ACER={ft_res['test']['at_val_tau']['acer']} | saved {adapter_file.name} ({adapter_mb} MB)")

    del model
    torch.cuda.empty_cache()
    return (zs_res, va_zs, te_zs), (ft_res, best_va_ft, te_ft)

In [ ]:
# 6. Chạy thí nghiệm cho cả 2 backbone (SigLIP2-Base & ViT-Base) + CNN nhẹ EfficientNet-B0
import timm

all_results = {}
val_preds = {}
test_preds = {}

# Backbone 1: SigLIP2-Base (king1oo1/deepfake-model)
(r_zs_sig, va_zs_sig, te_zs_sig), (r_ft_sig, va_ft_sig, te_ft_sig) = run_hf_experiment(
    tag="siglip2",
    model_id="king1oo1/deepfake-model",
    unfreeze_top_k=4,
    epochs=10,
    batch_size=64,
    lr_blocks=2e-5,
    lr_ln=1e-4,
    lr_head=3e-4,
)
all_results["zeroshot_siglip2"] = r_zs_sig
all_results["finetuned_siglip2"] = r_ft_sig
val_preds["zeroshot_siglip2"], test_preds["zeroshot_siglip2"] = va_zs_sig, te_zs_sig
val_preds["finetuned_siglip2"], test_preds["finetuned_siglip2"] = va_ft_sig, te_ft_sig

# Backbone 2: ViT-Base (prithivMLmods/Deep-Fake-Detector-v2-Model)
(r_zs_vit, va_zs_vit, te_zs_vit), (r_ft_vit, va_ft_vit, te_ft_vit) = run_hf_experiment(
    tag="vit",
    model_id="prithivMLmods/Deep-Fake-Detector-v2-Model",
    unfreeze_top_k=4,
    epochs=10,
    batch_size=64,
    lr_blocks=2e-5,
    lr_ln=1e-4,
    lr_head=3e-4,
)
all_results["zeroshot_vit"] = r_zs_vit
all_results["finetuned_vit"] = r_ft_vit
val_preds["zeroshot_vit"], test_preds["zeroshot_vit"] = va_zs_vit, te_zs_vit
val_preds["finetuned_vit"], test_preds["finetuned_vit"] = va_ft_vit, te_ft_vit

# Backbone 3 (Bonus CNN nhẹ siêu nhanh CPU): EfficientNet-B0 (timm)
print("\n==================== EFFICIENTNET-B0 (timm lightweight CNN) ====================")
set_seed(SEED)
cnn = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2, drop_rate=0.3).to(DEVICE)
inet_mean, inet_std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
dl_tr_cnn = DataLoader(RTFSCropDataset(LOCAL_DATA_DIR / "train", inet_mean, inet_std, 224, augment=True), batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
dl_va_cnn = DataLoader(RTFSCropDataset(LOCAL_DATA_DIR / "val",   inet_mean, inet_std, 224, augment=False), batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
dl_te_cnn = DataLoader(RTFSCropDataset(LOCAL_DATA_DIR / "test",  inet_mean, inet_std, 224, augment=False), batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

opt_cnn = torch.optim.AdamW(cnn.parameters(), lr=3e-4, weight_decay=0.03)
sch_cnn = torch.optim.lr_scheduler.CosineAnnealingLR(opt_cnn, T_max=12, eta_min=1e-6)
crit_cnn = nn.CrossEntropyLoss(weight=torch.tensor([2.0, 2.0/3.0], device=DEVICE), label_smoothing=0.05)

best_cnn_auc, best_cnn_sd, best_va_cnn = -1.0, None, None
for ep in range(1, 13):
    cnn.train()
    for x, y, _ in dl_tr_cnn:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, dtype=torch.long, non_blocking=True)
        opt_cnn.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE):
            loss = crit_cnn(cnn(x), y)
        loss.backward()
        opt_cnn.step()
    sch_cnn.step()
    va_s = predict_video_scores(cnn, dl_va_cnn, fake_idx=1, is_hf=False)
    va_auc = float(roc_auc_score(val_y_vid, [va_s[sid] for sid in df_val["sample_id"]]))
    if va_auc > best_cnn_auc:
        best_cnn_auc = va_auc
        best_va_cnn = va_s
        best_cnn_sd = {k: v.detach().cpu().clone() for k, v in cnn.state_dict().items()}
    print(f"    [EfficientNet-B0] ep {ep:02d}/12 | val_AUC={va_auc:.4f}{' *' if va_auc == best_cnn_auc else ''}")

cnn.load_state_dict(best_cnn_sd)
te_cnn = predict_video_scores(cnn, dl_te_cnn, fake_idx=1, is_hf=False)
cnn_tau, _ = find_optimal_threshold(val_y_vid, np.asarray([best_va_cnn[sid] for sid in df_val["sample_id"]]))
torch.save({"arch": "efficientnet_b0", "recommended_threshold": round(cnn_tau, 4), "state_dict": best_cnn_sd}, DRIVE_OUT_DIR / "m5_rtfs_efficientnet_b0.pt")
all_results["finetuned_efficientnet_b0"] = {
    "model_id": "timm/efficientnet_b0",
    "method": "Full Fine-tune CNN (L4 BF16, 4.0M params)",
    "val_tau": round(cnn_tau, 4),
    "val": evaluate_split(df_val, best_va_cnn, cnn_tau, with_ci=False),
    "test": evaluate_split(df_test, te_cnn, cnn_tau, with_ci=True),
}
val_preds["finetuned_efficientnet_b0"], test_preds["finetuned_efficientnet_b0"] = best_va_cnn, te_cnn

# Tổ hợp 50/50 Zero-shot và Fine-tuned (SigLIP2 + ViT)
for ens_key, k1, k2, desc in [
    ("zeroshot_ensemble", "zeroshot_siglip2", "zeroshot_vit", "Zero-shot Ensemble 50/50 (SigLIP2 + ViT)"),
    ("finetuned_ensemble", "finetuned_siglip2", "finetuned_vit", "Fine-tuned Ensemble 50/50 (SigLIP2 + ViT)"),
]:
    va_e = {sid: 0.5 * val_preds[k1][sid] + 0.5 * val_preds[k2][sid] for sid in df_val["sample_id"]}
    te_e = {sid: 0.5 * test_preds[k1][sid] + 0.5 * test_preds[k2][sid] for sid in df_test["sample_id"]}
    tau_e, _ = find_optimal_threshold(val_y_vid, np.asarray([va_e[sid] for sid in df_val["sample_id"]]))
    all_results[ens_key] = {
        "model_id": "king1oo1/deepfake-model + prithivMLmods/Deep-Fake-Detector-v2-Model",
        "method": desc,
        "val_tau": round(tau_e, 4),
        "val": evaluate_split(df_val, va_e, tau_e, with_ci=False),
        "test": evaluate_split(df_test, te_e, tau_e, with_ci=True),
    }
    val_preds[ens_key], test_preds[ens_key] = va_e, te_e

In [ ]:
# 7. Bảng tổng hợp E5 trên tập TEST giữ kín (200 video, 0 rò rỉ) & xuất biểu đồ báo cáo
import matplotlib.pyplot as plt

summary_rows = []
for name, res in all_results.items():
    t = res["test"]
    m = t["at_val_tau"]
    sl = t["slices"]
    summary_rows.append({
        "Model": name,
        "Val AUC": res["val"]["roc_auc"],
        "TEST AUC": t["roc_auc"],
        "95% CI": f"[{t['ci95'][0]:.3f}, {t['ci95'][1]:.3f}]",
        "EER": t["eer"],
        "Tau (val)": res["val_tau"],
        "Acc": m["accuracy"],
        "F1-macro": m["f1_macro"],
        "APCER": m["apcer"],
        "BPCER": m["bpcer"],
        "ACER": m["acer"],
        "InSwapper AUC": sl["inswapper_all"]["roc_auc"],
        "InSwap+Enh": sl["inswapper_with_enh"]["roc_auc"],
        "UniFace AUC": sl["uniface_all"]["roc_auc"],
        "UniFace+Enh": sl["uniface_with_enh"]["roc_auc"],
    })

df_summary = pd.DataFrame(summary_rows).sort_values("TEST AUC", ascending=False)
display(df_summary)

# Vẽ ROC Curves trên tập test
test_y = (df_test["label"] == "Deepfake").astype(int).to_numpy()
plt.figure(figsize=(8, 6), dpi=140)
for name in ["zeroshot_siglip2", "zeroshot_vit", "finetuned_efficientnet_b0", "finetuned_vit", "finetuned_siglip2", "finetuned_ensemble"]:
    ys = np.asarray([test_preds[name][sid] for sid in df_test["sample_id"]])
    fpr, tpr, _ = roc_curve(test_y, ys)
    auc_v = all_results[name]["test"]["roc_auc"]
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc_v:.4f})")
plt.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
plt.xlabel("False Positive Rate (BPCER)")
plt.ylabel("True Positive Rate (1 - APCER)")
plt.title("E5 Held-out Test ROC Curves — RTFS-800 (200 unseen videos)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DRIVE_OUT_DIR / "m5_e5_roc_curves.png")
plt.show()

# Lưu JSON + CSV kết quả về Google Drive
(DRIVE_OUT_DIR / "m5_e5_colab_benchmark.json").write_text(
    json.dumps(all_results, indent=2, ensure_ascii=False), encoding="utf-8"
)
pred_df = df_test[["sample_id", "label", "generator", "has_faceenhancer", "lighting", "gender"]].copy()
for k, d in test_preds.items():
    pred_df[k] = [round(d[sid], 6) for sid in df_test["sample_id"]]
pred_df.to_csv(DRIVE_OUT_DIR / "m5_e5_colab_predictions.csv", index=False)
print("Đã lưu toàn bộ adapter, JSON, CSV và biểu đồ tại:", DRIVE_OUT_DIR)

In [ ]:
# 8. Lưu, kiểm tra toàn vẹn & đóng gói toàn bộ Checkpoint + Kết quả E5 vào Google Drive
import hashlib, json, os, shutil, zipfile
from datetime import datetime
from pathlib import Path
import torch
from transformers import AutoModelForImageClassification

# Thư mục lưu cố định trên Google Drive (chỉnh tên nếu muốn)
FINAL_DRIVE_DIR = Path("/content/drive/MyDrive/AntiDeepFake-Face/m5_checkpoints_final")
FINAL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

expected_files = [
    "m5_rtfs_siglip2_adapter.pt",
    "m5_rtfs_vit_adapter.pt",
    "m5_rtfs_efficientnet_b0.pt",
    "m5_e5_colab_benchmark.json",
    "m5_e5_colab_predictions.csv",
    "m5_e5_roc_curves.png",
]

print(f"=== Kiểm tra & sao lưu checkpoint vào: {FINAL_DRIVE_DIR} ===")
saved_paths = []
manifest = {
    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "source_dir": str(DRIVE_OUT_DIR),
    "files": {},
}

for fname in expected_files:
    src = DRIVE_OUT_DIR / fname
    if not src.is_file():
        print(f"  [THIẾU] {src}")
        continue
    dst = FINAL_DRIVE_DIR / fname
    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)
    size_mb = dst.stat().st_size / (1024 * 1024)
    sha256 = hashlib.sha256(dst.read_bytes()).hexdigest()[:16]
    manifest["files"][fname] = {"size_mb": round(size_mb, 2), "sha256_16": sha256}
    saved_paths.append(dst)
    print(f"  [OK] {fname:32s} | {size_mb:6.2f} MB | sha256={sha256}")

# Kiểm chứng nạp lại adapter chính (m5_rtfs_siglip2_adapter.pt) trước khi tắt Colab
siglip_ckpt = FINAL_DRIVE_DIR / "m5_rtfs_siglip2_adapter.pt"
if siglip_ckpt.is_file():
    payload = torch.load(siglip_ckpt, map_location="cpu", weights_only=False)
    base_model = AutoModelForImageClassification.from_pretrained(payload["model_id"])
    cast_sd = {k: v.float() for k, v in payload["state_dict"].items()}
    _missing, unexpected = base_model.load_state_dict(cast_sd, strict=False)
    assert not unexpected, f"Lỗi unexpected keys: {unexpected[:5]}"
    with torch.inference_mode():
        dummy_logits = base_model(pixel_values=torch.randn(1, 3, 224, 224)).logits
        probs = torch.softmax(dummy_logits, dim=-1)[0]
    print(
        f"\n[VERIFY OK] Đã test nạp lại {siglip_ckpt.name} thành công! "
        f"keys={len(cast_sd)} | recommended_tau={payload.get('recommended_threshold')} | "
        f"dummy_fake_prob={probs[payload['fake_label_index']].item():.4f}"
    )
    del base_model

# Ghi manifest.json & đóng gói 1 file ZIP duy nhất để tải về Mac nhanh
manifest_path = FINAL_DRIVE_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

bundle_zip = FINAL_DRIVE_DIR / "m5_colab_deliverables.zip"
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in saved_paths + [manifest_path]:
        folder = "models" if p.suffix == ".pt" else "results"
        zf.write(p, arcname=f"{folder}/{p.name}")

os.sync()
print(f"\n[DONE] Đã lưu xong gói tải về 1 file tại: {bundle_zip} ({bundle_zip.stat().st_size / (1024*1024):.1f} MB)")
print("Gợi ý: Tải file m5_colab_deliverables.zip về máy Mac và giải nén thẳng vào thư mục gốc AntiDeepFake-Face/!")

### Hướng dẫn đưa mô hình đã train về máy local (`AntiDeepFake-Face`)
1. Tải file **`m5_rtfs_siglip2_adapter.pt`** từ `MyDrive/AntiDeepFake-Face/m5_colab_outputs/` và đặt vào thư mục **`models/m5_rtfs_siglip2_adapter.pt`** trên máy Mac.
2. `src/deepfake/predictor.py` sẽ **tự động nhận diện và nạp adapter này** đè lên `king1oo1/deepfake-model` (giữ nguyên tốc độ CPU `11–14 FPS`!).
3. Copy `m5_e5_colab_benchmark.json`, `m5_e5_colab_predictions.csv` và `m5_e5_roc_curves.png` vào `results/` để điền số liệu thật vào `docs/report/final_report.md` (mục E5).
4. Chạy lại `.venv/bin/python -m pytest -v tests/test_deepfake_m5.py tests/test_anti_spoof_deepfake.py` để xác nhận xanh 100%.